# Fine-tune PaddleOCR-VL-1.6 with LoRA

PaddleOCR-VL-1.6 is a 0.9B vision-language model. This notebook adapts it to text-line recognition on CORD receipts with LoRA, on a free T4, and measures it on the same line crops as the PP-OCRv6 notebook.

**Runtime → Change runtime type → T4 GPU.**

## 1. Set up and prepare the line crops

In [ ]:
%env TQDM_DISABLE=1
!git clone -q https://github.com/Dmitry543/Fine-Tuning-OCR-Model-with-PaddleOCR.git
%cd Fine-Tuning-OCR-Model-with-PaddleOCR
!pip install -q --progress-bar off -r vl/requirements.txt pyarrow requests opencv-python-headless
!python tools/prepare_data.py cord --out data/cord

## 2. Measure the stock model

In [ ]:
!python vl/evaluate.py --name vl_stock

## 3. Train the LoRA adapter

One epoch over the ~13,000 training lines. Only 0.7% of the weights (the adapters) are trained; the vision encoder is frozen.

In [ ]:
!python vl/train_lora.py --out output/vl_lora

## 4. Measure again

In [ ]:
!python vl/evaluate.py --name vl_lora --adapter output/vl_lora
import json
for name in ["vl_stock", "vl_lora"]:
    r = json.load(open(f"results/cord/{name}.json"))["recognition"]
    print(f"{name:10s} accuracy {100 * r['accuracy']:.1f}%  CER {100 * r['cer']:.2f}%  {r['ms_per_line']:.0f} ms/line")

## 5. Use it

The adapter is a few MB. Load it on top of the base model:

In [ ]:
import torch, sys
sys.path.insert(0, "vl")
from peft import PeftModel
from transformers import AutoModelForImageTextToText, AutoProcessor
from data import MODEL, generation_batch

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.float16).to("cuda").eval()
model = PeftModel.from_pretrained(model, "output/vl_lora").merge_and_unload()

batch = generation_batch(processor, ["data/cord/rec/test/test_0000_001.jpg"]).to("cuda")
out = model.generate(**batch, max_new_tokens=64)
print(processor.decode(out[0, batch["input_ids"].shape[1]:], skip_special_tokens=True))